# Serving the final model on the CPU
Hosting NLLB-200 1.3B on a GPU is the expensive part of serving it. This notebook merges the adapter
of the final model (§7.4 of `sicilian_nmt.ipynb`) into the base model, converts it to CTranslate2
with int8 weights (`experiments/serving/ct2_export.py`, saved to `nllb-final-ct2-int8` on Drive)
and measures it on the CPU of the runtime with 1, 2 and 4 threads: time to load, resident memory
and latency of one sentence at a time (beam 5) on 100 English sentences of our test set translated
into Sicilian. It then translates the 8 test files of §7.9 with int8 weights and scores them next
to the fp16 GPU hypotheses of that cell, raw and in the tokenized space of Wdowiak, so that the
cost of the quantization is measured on the same lines. Results go to `results_ct2.json`, and a
finished step is skipped. The merge needs about 12 GB of RAM, so a GPU runtime (for its memory
and its CPUs, not for the GPU) is the safe choice.

In [ ]:
!pip -q install transformers==4.57.6 peft==0.17.1 ctranslate2 sentencepiece sacrebleu
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
!git -C SicilianNMT pull -q
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'
sys.path.append('SicilianNMT/experiments/serving')
import ct2_export
print('ready')

In [ ]:
# --- CTranslate2 int8 export of the final model and CPU benchmark ---
from glob import glob
ep = json.load(open(f'{OUT}/results_final.json'))['epoch']          # the epoch chosen on validation
ck = sorted(glob(f'{OUT}/nllb-lora-final-trainer/checkpoint-*'), key=lambda c: int(c.rsplit('-', 1)[1]))[ep - 1]
CT2 = ct2_export.export(ck, f'{OUT}/nllb-final-ct2-int8')
res = ct2_export.bench(CT2, f'{OUT}/translations', f'{OUT}/results_ct2.json')
print('checkpoint:', ck)
print(res['cpu'], res['cpu_count'], 'cpus, model', res['model_mb'], 'MB')
for r in res['runs']:
    print({k: v for k, v in r.items() if k != 'quality'})
for k, v in res['runs'][-1]['quality'].items():
    print(k, v)